# // Pip Install all Required Libraries //

The cell below installs extra required python libraries

In [ ]:
!pip install vedo scikit-image h5py tifffile networkx

# // Imports //

The cell below imports all the required libraries for this tutorial

In [ ]:
import sys
import os
from pathlib import Path

# --- CRITICAL FIX: MUST BE SET BEFORE NUMPY IMPORT ---
# Force all math libraries to use 1 thread to stop Docker freezing
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['NUMEXPR_NUM_THREADS'] = '1'
# -----------------------------------------------------

import numpy as np # Now truly safe to import

# image_proc_funcs.py is a helper file next to this notebook -- not part of the libcuflynx
# package -- so it is loaded from its own path rather than by editing the import path.
import importlib.util

this_dir = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
candidates = [this_dir / "image_proc_funcs.py",
              this_dir / "tutorial" / "interactive" / "image_proc_funcs.py",
              Path.cwd() / "image_proc_funcs.py",
              Path.cwd() / "tutorial" / "interactive" / "image_proc_funcs.py"]
helper_file = next((c for c in candidates if c.exists()), None)
if helper_file is None:
    raise FileNotFoundError("could not find image_proc_funcs.py; run this notebook from "
                            "tutorial/interactive or from the repository root")

spec = importlib.util.spec_from_file_location("image_proc_funcs", helper_file)
image_proc_funcs = importlib.util.module_from_spec(spec)
sys.modules["image_proc_funcs"] = image_proc_funcs
spec.loader.exec_module(image_proc_funcs)

IlastikClassifier = image_proc_funcs.IlastikClassifier
run_image_to_model = image_proc_funcs.run_image_to_model

# Force Matplotlib to not use a window
os.environ['MPLBACKEND'] = 'Agg'

# // Setting up Required Filepaths //

The cell below initialises all required file paths for the tutorial.

In [ ]:
# get dir where this file is. This should work locally or in Docker
this_dir = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
CA_root = Path(__file__).resolve().parent.parent.parent if "__file__" in globals() else Path.cwd().parent.parent
resources_path = CA_root / Path("resources")

# libcuflynx is imported from the installed package; nothing goes on the import path.

#################################################
### // Initialise target image files paths // ###
#################################################

image_path_0 = CA_root / Path("tutorial/interactive/resources/image_process/batch_process_output_folder/image_0_seg.h5") # Image 0
image_path_1 = CA_root / Path("tutorial/interactive/resources/image_process/batch_process_output_folder/image_1_seg.h5") # Image 1
image_path_2 = CA_root / Path("tutorial/interactive/resources/image_process/batch_process_output_folder/image_2_seg.h5") # Image 2
image_path_list = np.array([image_path_0, image_path_1, image_path_2])

#####################################
### // Initialise image target // ###
#####################################

### TODO: User to edit image_selection_index depending on which image (Image 0, Image 1, or Image 2) they would like to process.
image_selection_index = 0 ### Change value to 0, 1, or 2 depending on which image you would like to input into the pipeline
target_image_path = image_path_list[image_selection_index]

################################################
### // Initialise file paths for pipeline // ###
################################################

ilastik_path = Path("/opt/ilastik/run_ilastik.sh") ### TODO: Delete/uncomment after docker integration is working
model_path = CA_root / Path("tutorial/interactive/resources/segmentation_model.ilp")
input_batch_processing_path = CA_root / Path("tutorial/interactive/resources/image_process/batch_process_input_folder")
output_batch_processing_path = CA_root / Path("tutorial/interactive/resources/image_process/batch_process_output_folder")

print('paths set')

In [ ]:
# Download and unzip image resources from Dropbox
# Replace the URL with your Dropbox share link (ensure it uses ?dl=1)
import urllib.request
import zipfile

resources_dir = CA_root / Path("tutorial/interactive/resources")
resources_dir.mkdir(parents=True, exist_ok=True)

#TODO set this cell up for figshare rather than dropbox to make it more robust.

DROPBOX_ZIP_URL = "https://www.dropbox.com/scl/fi/jjvg4hckxgvi7dr50r456/image_process.zip?rlkey=45ydpi1i65vvixzmcv69u7bq2&st=numgvmlz&dl=1"
zip_path = resources_dir / "image_process.zip"

print("Downloading to", zip_path)
try:
    urllib.request.urlretrieve(DROPBOX_ZIP_URL, zip_path)
except Exception as err:
    # Docker images sometimes lack CA certs; fallback to unverified SSL.
    import ssl
    print("Retrying download without SSL verification:")
    unverified_ctx = ssl._create_unverified_context()
    with urllib.request.urlopen(DROPBOX_ZIP_URL, context=unverified_ctx) as resp, open(zip_path, "wb") as out_file:
        out_file.write(resp.read())

print("Unzipping...")
with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(resources_dir)

print("Done.")

# Image to Hemodynamics Model

This template walks through turning a microvasculature confocal microscopy image into a hemodynamics model, then exploring constriction (sympathetic activity) effects in PhLynx and WebOpenCOR.

## 1. Open ImageJ and visualize the image

0. JUMP-FORWARD: Run "2. Convert Image ... " cell then come back to this because converting takes 10 minutes to run. By the time you finish this section it should be finished!

1. Launch ImageJ (or Fiji).


In [ ]:
# Launch ImageJ/Fiji (requires X11 forwarding)
# On Linux OS, or Windows uncomment and run the line below
!/opt/fiji/fiji-linux-x64
# On Mac OS, uncomment and run the line below 
# !ln -sf /opt/fiji/java/linux64 /opt/fiji/java/linux-amd64 2>/dev/null; ln -sf /opt/fiji/java/linux64 /opt/fiji/java/linux-x64 2>/dev/null; export CONDA_PREFIX=/opt/conda CONDA_ROOT=/opt/conda CJDK_CACHE_DIR=/tmp XDG_CACHE_HOME=/tmp HOMEBREW_PREFIX=/tmp MAMBA_ROOT_PREFIX=/tmp JAVA_HOME=/opt/fiji/java/linux64; /opt/fiji/fiji-linux-x64

2. Open the source image: `File -> Open...` `circulatory_autogen/tutorial/interactive/resources/image_process/raw_image_files/Zstack1_Animal2_NG2_dsRed_CD31_647_GLUT_15042025/C2-Zstack1_Animal2_NG2_dsRed_CD31_647_GLUT_15042025_vessels.tif`
3. Verify the scale and orientation are correct (Image -> ShowInfo)
4. (Optional) Adjust contrast (Image -> Adjust -> Brightness/Contrast) or apply filters to clarify vessel boundaries.
When adjusting contrast, the changes are only superficial until apply is selected within Brightness/Contrast. It is generally inadvisable to permanently apply contrast changes and this should be used cosmetically).
In turns of filters, background subtract (Process -> Subtract Background) is a commonly used methodology to remove background noise (https://imagejdocu.list.lu/gui/process/subtract_background). Commonly used filters for vascular analysis include Guassian blur 3D (Process -> Filters -> Gaussian Blur 3D) and Frangi Vesselness (Process -> Filters -> Frangi Vesselness). These processes (amongst others) can improve vessel segmentation downstream, but classifiers should be trained on datasets using a chosen combination of image processing, and a chosen pipeline should be consistent amongst datasets that are directly compared.
5. (Optional) View image in 3D using Plugins -> BigDataViewer -> Open Current Image
6. (Optional) "Z-Project" to produce a projected 2D image from the 3D image series. This is typically how images are displayed within published papers in order to simplify the data (Image -> Stacks -> Z-Project -> Projection Type = Max intensity).
7. If you did any of the optional steps, close all open windows (without saving) and reopen the source image before proceeding. `File -> Open...` `circulatory_autogen/tutorial/interactive/resources/image_process/raw_image_files/Zstack1_Animal2_NG2_dsRed_CD31_647_GLUT_15042025/C2-Zstack1_Animal2_NG2_dsRed_CD31_647_GLUT_15042025_vessels.tif`

---- The instructions in this cell lets you see the vessel geometry that will be converted into a model but is not required for the next steps. ----

## 2. Convert image to `module_array.csv` and `parameters.csv`

Next we will run Python to process the image and generate two outputs:

- `module_array.csv` describing the vessel network connectivity and geometry
- `parameters.csv` defining model parameters

The conversion functions are not available yet, so the next cell contains placeholders.

In [ ]:
#############################################################
### // Specify sub-volume (<= 0.1) of image to process // ###
#############################################################

### TODO: User to specify desired sub-volume. Ensure the value is <= 0.1 to avoid long processing times.
sub_volume = 0.1

##########################
### // Run pipeline // ###
##########################

run_ilastik_batch_processing = True ### TODO: Change to True if you want to use in-python Ilastik image segmentation

run_image_to_model(target_image_path, resources_path, ilastik_path, model_path,
                   input_batch_processing_path, output_batch_processing_path, sub_volume, run_ilastik_batch_processing)

print("Wrote:", str(CA_root / Path("tutorial/interactive/resources/user_output/image_to_model_module_array.csv")))
print("Wrote:", str(CA_root / Path("tutorial/interactive/resources/user_output/image_to_model_parameters.csv")))

## 3. Open the CSV files in PhLynx

1. Go to https://www.phlynx.com
2. Create a new project.
3. Import `module_array.csv` and `parameters.csv`.
4. Confirm the network and parameters are shown correctly.

## 4. Simulate in WebOpenCOR from PhLynx

1. From PhLynx, open the model in WebOpenCOR.
2. Run a baseline simulation.
3. Verify that pressures and flows are reasonable.

## 5. Add a sympathetic constriction module in PhLynx

1. Back in PhLynx, create a module (search new_module in modules) that modifies capillary-pericyte radius based on application of alpha blocker.

3. Use:

- constricted diameter $d_c = 3.83 um$

- fully relaxed diameter $d_r = 4.32 um$

- pericyte resistance modulator $R_{mod} - Js/m6$
  
3. Assume the baseline resistance of the vessel section is

4. Let's now assume one of our vessels is covered by a pericyte that constricts (the assumptions can be improved).

$$ R_{mod} = (d_r/d)^4$$

For linearly changing diameter with x location.

5. Attach this module to a vessel to modify its resistance.

   - In the vessel module you are connecting to edit cellml and include a variable R_mod, which multiplies the RHS of the R equation in the vessel module.
   - In the vessel module you are connecting to click "Edit Parameters" and set R_mod to a boundary condition
   - Create a corresponding port on both modules, R_mod port so the vessel can see the modulated resistance.

7. Save the updated project.
8. Export to CellML




EXTRA COMPLEXITY BELOW: 

- For pericytes, it can be mathematically shown that

- $$ R_{mod} = \frac{L_c - L_p}{L_c} + \frac{L_p}{L_c} \frac{(1-(d/d_r)^{(3+n)})}{(d_r/d-1)(3+n)}$$

- For linearly changing radius with x location.

- pericyte length of effect $L_p = 30 um$

- capillary section length of effect $L_c$

- experimentally fit capillary factor accounting for viscocity effects at small diameter $n = 1.647$


## 6. Add a slider in WebOpenCOR to demonstrate flow effects of pericyte constriction

OpenCOR is a fantastic tool for investigating physiological effects to changes in parameters quickly!

1. Open the updated model in OpenCOR.
2. Create plots for flow at the vessel where you included a pericyte.
3. Create a slider (simulation_input) called constriction that will be used to reduce diameter.
5. Sweep the slider across its range and observe changes in pressure/flow at various locations.



## Conclusion

1. We can generate a working hemodynamics model from a microvasculature image with Circulatory_Autogen and PhLynX.
2. PhLynx allows us to create modules that can be easily coupled together.
3. These modules will can be saved in a repo, ensuring repeatable, reuasable and shareable modelling!
4. We can easily visually investigate changes in parameters and their physiological response.
5. NEXT STEPS: To do automated quantifiable analysis, we want to be able to script model generation and calibration, which we will see in Tutorial 5.